## Install Required Libraries

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
!pip install -q datasets transformers evaluate sentence-transformers openai pandas numpy tqdm python-dotenv  google-generativeai requests tqdm -q sacrebleu rouge_score bert_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.7 MB/s eta 0:00:00


In [3]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 733.5/733.5 kB 13.9 MB/s eta 0:00:00


In [4]:
!pip install pandas datasets anthropic tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 8.7 MB/s eta 0:00:00


In [5]:
import os
import csv
import json
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import evaluate
from sentence_transformers import SentenceTransformer, util
from peft import PeftModel

# Other Packages
import pandas as pd
import numpy as np
import os
from huggingface_hub import login
import getpass
from google.colab import userdata
from tqdm.auto import tqdm
import os
import csv
import os, json, time, math
from tqdm import tqdm
import google.generativeai as genai
import re

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


### Display Settings

In [6]:
pd.set_option('display.max_rows', None)        # Show all rows
pd.set_option('display.max_columns', None)     # Show all columns
pd.set_option('display.width', None)           # Don't wrap to next line
pd.set_option('display.max_colwidth', None)    # Show full column content

### Hugginface Login

In [7]:
HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in using environment variable")
else:
    print("No HF_TOKEN found")

Logged in using environment variable


## Gemini Login

In [8]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

### Mount Google Drive

In [10]:
from google.colab import drive

# mount drive
drive.mount('/content/drive')

Mounted at /content/drive


### Load the test split and choose sample size

In [ ]:
dataset_name = "Lakshan2003/customer-support-context-summary-50k"
split_name   = "test"
num_samples  =  None

test_dataset = load_dataset(dataset_name, split=split_name)
if num_samples is not None:
    test_dataset = test_dataset.select(range(num_samples))  # Keep original order

print(f"Loaded {len(test_dataset)} test samples")

README.md:   0%|          | 0.00/826 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/5.82M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/11.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/35000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loaded 10000 test samples


### Generation Settings

In [ ]:
model_name   = "gemini-2.5-flash"
results_dir  = f"/content/drive/MyDrive/fyp-2025/Datasets/ContextSummaryTestData/{model_name}"
results_path = os.path.join(results_dir, f"results_{model_name}.csv")
os.makedirs(results_dir, exist_ok=True)

In [ ]:
def build_user_prompt(example):
    return (
        f"Conversation History:\n"
        f"{example.get('conversation_history','')}"
    ).strip()

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
# Detect previously processed IDs
if os.path.exists(results_path) and os.path.getsize(results_path) > 0:
    existing_df = pd.read_csv(results_path, usecols=["conversation_id"])
    processed_ids = set(existing_df["conversation_id"].astype(str))
    header_written = True
    print(f"Resuming from existing file: {len(processed_ids)} already processed.")
else:
    processed_ids = set()
    header_written = False
    print("Starting fresh run; no existing results found.")

Starting fresh run; no existing results found.


In [ ]:
import time

In [ ]:
batch_size = 100
batch_rows = []

# Main loop
for idx, example in enumerate(tqdm(test_dataset, desc="Evaluating Gemini 2.5")):
    conv_id = str(example.get("conversation_id", "")).strip()
    if not conv_id or conv_id in processed_ids:
        continue  # skip already processed ones

    # Prompts
    system_prompt = (example.get("summarization_prompt", "") or "").strip()

    user_prompt = (build_user_prompt(example) or "").strip()
    full_prompt = f"{system_prompt}\n\n{user_prompt}" if system_prompt else user_prompt

    try:
        # API call
        time.sleep(1)

        response = client.models.generate_content(
            model=model_name,
            contents=[full_prompt],
            config=types.GenerateContentConfig(
                temperature=0.7,
                top_p=0.9,
                max_output_tokens=256,
                thinking_config=types.ThinkingConfig(thinking_budget=0)
            ),
        )

        text_raw = getattr(response, "text", "") or ""
        generated_text = re.sub(r"\s+", " ", text_raw.strip())

    except Exception as e:
        print(f"[Gemini] Error at row {idx} (conv_id={conv_id}): {e}")
        print("Stopping execution to prevent data loss. You can safely re-run this script later; it will resume.")
        # Save partial progress before stopping
        if batch_rows:
            pd.DataFrame(batch_rows).to_csv(results_path, mode="a", index=False, header=not header_written)
            print(f"Saved partial batch of {len(batch_rows)} before stopping.")
        raise SystemExit(e)

    # Append and mark as processed
    batch_rows.append({
        "conversation_id": conv_id,
        "instruction": example.get("instruction", ""),
        "conversation_history": example.get("conversation_history", ""),
        "summarization_prompt": user_prompt,
        "history_summary": example.get("history_summary", ""),
        "client_question": example.get("client_question", ""),
        "agent_answer": example.get("agent_answer", ""),
        "refined_agent_answer": example.get("refined_agent_answer", ""),
        "generated_summary": generated_text,
    })
    processed_ids.add(conv_id)

    # Write in batches
    if len(batch_rows) >= batch_size:
        pd.DataFrame(batch_rows).to_csv(results_path, mode="a", index=False, header=not header_written)
        header_written = True
        print(f"[Gemini] Wrote batch of {batch_size} rows; total processed: {len(processed_ids)}")
        batch_rows = []

# Write remaining
if batch_rows:
    pd.DataFrame(batch_rows).to_csv(results_path, mode="a", index=False, header=not header_written)
    print(f"[Gemini] Wrote final batch of {len(batch_rows)} rows. Total processed: {len(processed_ids)}")

print("[Gemini] Inference completed and results saved:", results_path)

Evaluating Gemini 2.5:   1%|          | 100/10000 [03:01<5:04:37,  1.85s/it]

[Gemini] Wrote batch of 100 rows; total processed: 100


Evaluating Gemini 2.5:   2%|▏         | 200/10000 [06:06<5:29:19,  2.02s/it]

[Gemini] Wrote batch of 100 rows; total processed: 200


Evaluating Gemini 2.5:   3%|▎         | 300/10000 [09:08<4:56:45,  1.84s/it]

...

[Gemini] Wrote batch of 100 rows; total processed: 9800


Evaluating Gemini 2.5:  99%|█████████▉| 9900/10000 [5:01:12<02:47,  1.67s/it]

[Gemini] Wrote batch of 100 rows; total processed: 9900


Evaluating Gemini 2.5: 100%|██████████| 10000/10000 [5:04:11<00:00,  1.83s/it]

[Gemini] Wrote batch of 100 rows; total processed: 10000
[Gemini] Inference completed and results saved: /content/drive/MyDrive/fyp-2025/Datasets/ContextSummaryTestData/gemini-2.5-flash/results_gemini-2.5-flash.csv


In [ ]:
import pandas as pd
from datasets import Dataset

# Load CSV
df = pd.read_csv(results_path)

# Convert to Dataset
dataset = Dataset.from_pandas(df)

# Define repo name with model + use case
repo_id = "Lakshan2003/gemini-2.5-flash-customerservice-context-summarization-evaldata"

# Push to Hub
dataset.push_to_hub(repo_id, private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  30%|##9       | 5.78MB / 19.6MB            

CommitInfo(commit_url='https://huggingface.co/datasets/Lakshan2003/gemini-2.5-flash-customerservice-context-summarization-evaldata/commit/e3f5ef896629b4d1f38a57b8146acf5762721a5d', commit_message='Upload dataset', commit_description='', oid='e3f5ef896629b4d1f38a57b8146acf5762721a5d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Lakshan2003/gemini-2.5-flash-customerservice-context-summarization-evaldata', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Lakshan2003/gemini-2.5-flash-customerservice-context-summarization-evaldata'), pr_revision=None, pr_num=None)

### Lexical Overlap Calculation

In [ ]:
df = pd.read_csv(results_path)
preds = df["generated_summary"].tolist()
refs  = df["history_summary"].tolist()

### Bleu Score Calculation

In [ ]:
# Lexical overlap metrics
bleu   = evaluate.load("bleu").compute(predictions=preds, references=[[r] for r in refs])["bleu"]

bleu

0.26521277004523747

### Google Bleu Score Calculation

In [ ]:
gbleu  = evaluate.load("google_bleu").compute(predictions=preds, references=refs)["google_bleu"]

gbleu

0.29766804594100854

### Meteor Score Calculation

In [ ]:
meteor = evaluate.load("meteor").compute(predictions=preds, references=refs)["meteor"]

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
print(meteor)

0.47298112594713504


### Rougue Score Calculation

In [ ]:
rouge  = evaluate.load("rouge").compute(predictions=preds, references=refs)

rouge

{'rouge1': np.float64(0.5911824790600586),
 'rouge2': np.float64(0.33542101662815105),
 'rougeL': np.float64(0.45112598889423683),
 'rougeLsum': np.float64(0.4510212094073271)}

### Summarised Lexical Overlap Results

In [ ]:
import pandas as pd

# Build a summary table
metrics_table = pd.DataFrame([
    {"metric": "BLEU",         "score": bleu},
    {"metric": "Google BLEU",  "score": gbleu},
    {"metric": "METEOR",       "score": meteor},
    {"metric": "ROUGE-1",      "score": rouge["rouge1"]},
    {"metric": "ROUGE-2",      "score": rouge["rouge2"]},
    {"metric": "ROUGE-L",      "score": rouge["rougeL"]},
])

In [ ]:
metrics_table

,metric,score
0,BLEU,0.265213
1,Google BLEU,0.297668
2,METEOR,0.472981
3,ROUGE-1,0.591182
4,ROUGE-2,0.335421
5,ROUGE-L,0.451126


In [ ]:
from datasets import Dataset

repo = "Lakshan2003/Lexical-Overlap-Results-gemini-2.5-flash-customerservice-context-summary"

hf_dataset = Dataset.from_pandas(metrics_table.reset_index(drop=True))

# Push to the hub
hf_dataset.push_to_hub(repo)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 1.35kB / 1.35kB            

CommitInfo(commit_url='https://huggingface.co/datasets/Lakshan2003/Lexical-Overlap-Results-gemini-2.5-flash-customerservice-context-summary/commit/fa12f82e16ef2ac4fd0b37ab045619320bf0fb71', commit_message='Upload dataset', commit_description='', oid='fa12f82e16ef2ac4fd0b37ab045619320bf0fb71', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Lakshan2003/Lexical-Overlap-Results-gemini-2.5-flash-customerservice-context-summary', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Lakshan2003/Lexical-Overlap-Results-gemini-2.5-flash-customerservice-context-summary'), pr_revision=None, pr_num=None)

## Semantic Similarity Metrics

In [ ]:
from datasets import load_dataset

ds = load_dataset("csv", data_files=results_path, split="train")
ds

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['conversation_id', 'instruction', 'conversation_history', 'summarization_prompt', 'history_summary', 'client_question', 'agent_answer', 'refined_agent_answer', 'generated_summary'],
    num_rows: 10000
})

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Generate embeddings for Groundtruth and generated answer

In [ ]:
def embed_batch(batch):
    gen = [str(x) if x is not None else "" for x in batch["generated_summary"]]
    gt  = [str(x) if x is not None else "" for x in batch["history_summary"]]

    # normalize_embeddings=True => unit vectors, so cosine = dot product
    gen_emb = model.encode(gen, convert_to_numpy=True, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    gt_emb  = model.encode(gt,  convert_to_numpy=True, normalize_embeddings=True, batch_size=64, show_progress_bar=False)

    # store as lists
    return {
        "generated_summary_embedding_transformers": [e.tolist() for e in gen_emb],
        "ground_truth_embedding_transformers": [e.tolist() for e in gt_emb],
    }

ds = ds.map(embed_batch, batched=True, batch_size=256)
ds

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset({
    features: ['conversation_id', 'instruction', 'conversation_history', 'summarization_prompt', 'history_summary', 'client_question', 'agent_answer', 'refined_agent_answer', 'generated_summary', 'generated_summary_embedding_transformers', 'ground_truth_embedding_transformers'],
    num_rows: 10000
})

In [ ]:
import numpy as np

def cosine_batch(batch):
    gen = np.array(batch["generated_summary_embedding_transformers"], dtype=np.float32)
    gt  = np.array(batch["ground_truth_embedding_transformers"], dtype=np.float32)

    # cosine = sum(gen * gt) (using normalized embeddings)
    cos = (gen * gt).sum(axis=1)
    return {"cosine_similarity": cos.tolist()}

ds = ds.map(cosine_batch, batched=True, batch_size=1024)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

### BERT SCORE

In [ ]:
import evaluate
bertscore = evaluate.load("bertscore")

def add_bertscore(batch):
    preds = [str(x) if x else "" for x in batch["generated_summary"]]
    refs  = [str(x) if x else "" for x in batch["history_summary"]]
    result = bertscore.compute(predictions=preds, references=refs, lang="en")
    return {
        "bertscore_precision": result["precision"],
        "bertscore_recall": result["recall"],
        "bertscore_f1": result["f1"],
    }

ds = ds.map(add_bertscore, batched=True, batch_size=64)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### BART SCORE

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

bart_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn").to(device)
bart_model.eval()

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [ ]:
def compute_bartscore(sources, targets):
    scores = []
    for src, tgt in zip(sources, targets):
        inputs = bart_tokenizer(src, return_tensors="pt", truncation=True, max_length=512).to(device)
        with bart_tokenizer.as_target_tokenizer():
            labels = bart_tokenizer(tgt, return_tensors="pt", truncation=True, max_length=512).to(device)
        with torch.no_grad():
            loss = bart_model(**inputs, labels=labels["input_ids"]).loss
        scores.append(-loss.item())  # negative loss = BARTScore
    return scores

def add_bartscore(batch):
    preds = [str(x) if x else "" for x in batch["generated_summary"]]
    refs  = [str(x) if x else "" for x in batch["history_summary"]]
    ref_hyp = compute_bartscore(refs, preds)
    hyp_ref = compute_bartscore(preds, refs)
    mean_score = [(a + b) / 2 for a, b in zip(ref_hyp, hyp_ref)]
    return {
        "bartscore_ref_hyp": ref_hyp,
        "bartscore_hyp_ref": hyp_ref,
        "bartscore_mean": mean_score,
    }

In [ ]:
ds = ds.map(add_bartscore, batched=True, batch_size=128)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [ ]:
df = ds.to_pandas()

In [ ]:
print("Summary:")
print(f"Mean cosine similarity : {df['cosine_similarity'].mean():.4f}")
print(f"Mean BERTScore F1      : {df['bertscore_f1'].mean():.4f}")
print(f"Mean BARTScore (mean)  : {df['bartscore_mean'].mean():.4f}")

Summary:
Mean cosine similarity : 0.8656
Mean BERTScore F1      : 0.9194
Mean BARTScore (mean)  : -2.1386


In [ ]:
import os
import pandas as pd

MODEL_NAME = "gemini-2.5-flash"
SAVE_DIR = "/content/drive/MyDrive/fyp-2025/Datasets/Contex_Summary_Semantic_Lexical_Similarity_Results"

metrics_rows = [
    {"model": MODEL_NAME, "metric": "BLEU",        "value": float(bleu)},
    {"model": MODEL_NAME, "metric": "Google BLEU", "value": float(gbleu)},
    {"model": MODEL_NAME, "metric": "METEOR",      "value": float(meteor)},
    {"model": MODEL_NAME, "metric": "ROUGE-1",     "value": float(rouge["rouge1"])},
    {"model": MODEL_NAME, "metric": "ROUGE-2",     "value": float(rouge["rouge2"])},
    {"model": MODEL_NAME, "metric": "ROUGE-L",     "value": float(rouge["rougeL"])},
    {"model": MODEL_NAME, "metric": "Cosine Similarity (mean)", "value": float(df["cosine_similarity"].mean())},
    {"model": MODEL_NAME, "metric": "BERTScore F1 (mean)",      "value": float(df["bertscore_f1"].mean())},
    {"model": MODEL_NAME, "metric": "BARTScore (mean)",         "value": float(df["bartscore_mean"].mean())},
]

metrics_table = pd.DataFrame(metrics_rows)

os.makedirs(SAVE_DIR, exist_ok=True)
csv_path = f"{SAVE_DIR}/metrics_{MODEL_NAME}.csv"
metrics_table.to_csv(csv_path, index=False)

print(csv_path)

/content/drive/MyDrive/fyp-2025/Datasets/Contex_Summary_Semantic_Lexical_Similarity_Results/metrics_gemini-2.5-flash.csv


### LLM as a judge Evaluation

In [11]:
import os
import time
import json
import pandas as pd
from anthropic import Anthropic

#### Claude API setup

In [13]:
from google.colab import userdata
claude_api = userdata.get('claude_api')

In [14]:
anthropic = Anthropic(api_key=claude_api)

In [15]:
# Configuration
original_csv = "/content/drive/MyDrive/context_summarization_eval_samples/Gemini-25-Flash_1000_samples.csv"

processed_folder = "/content/drive/MyDrive/fyp-2025/Datasets/ContextSummarizationLLMJudge/"
processed_csv = processed_folder + "Gemini-25-Flash_context_summarization_eval.csv"

batch_size = 50
batch_pause = 1.5  # seconds

#### EVALUATION PROMPT TEMPLATE (outside loop)

In [16]:
EVAL_PROMPT = """
You are an expert evaluator specializing in dialogue summarization for customer-service interactions.
Evaluate the Generated Context Summary using the Original Multi-Turn Conversation as reference,
along with the Full Conversation History provided as additional background.
Your task is to assess how accurately, faithfully and clearly the Generated Context Summary
represents the key information from the conversation.

Note:
The Generated Context Summary is the model output being evaluated.
The Full Conversation History is provided only as supporting reference to help you better
understand the original interaction in detail.

Context Inputs:
Full Conversation History: {conversation_history}
Generated Context Summary: {generated_summary}

Evaluation Criteria and Scoring (1-5 each):

1. Information Accuracy

This measures how correctly the summary captures the important factual details from the conversation,
such as names, account numbers, dates, amounts, actions taken, verification steps, and follow-up commitments.

Rating Scale

1 = Important facts are missing or incorrect. The summary does not reflect the conversation properly.
2 = Some correct details are included, but several important facts are missing or wrong.
3 = Most important facts are correct, but a few details are missing. Overall meaning is still understandable.
4 = Almost all important facts are correct with only very small missing details.
5 = All key facts from the conversation are clearly and correctly included with no mistakes.


2. Structural Clarity

This measures how clearly the summary is organized. It checks whether the client issue,
conversation status, and key actions are presented in a clear and logical way.

Rating Scale

1 = The summary is confusing and poorly organized. The issue and status are hard to understand.
2 = Some structure exists, but the main issue or status is not clearly explained.
3 = The summary is generally understandable, but some parts are not clearly presented.
4 = The summary is well organized. The issue, status, and actions are easy to identify.
5 = The summary is very clear and logically structured. All parts are easy to read and understand.


3. Faithfulness

This measures whether the summary only includes information from the original conversation and does not add new or incorrect details.

Rating Scale

1 = The summary contains incorrect or invented information that was not in the conversation.
2 = Some parts include unsupported assumptions or slightly incorrect interpretations.
3 = Mostly faithful to the conversation, but a small amount of unsupported detail may appear.
4 = Very faithful to the conversation with only minimal inferred details that do not change the meaning.
5 = Completely faithful. The summary only contains information that appears in the conversation with no added or distorted content.

Return your answer as valid JSON only.
Do not include any explanation, commentary, additional text, or markdown formatting.
Output must contain JSON only and nothing else. All the below keys and their judgement score
should be included in your json response. Strictly follow only below json output. Always provide
the score for all tasks in the json.

Output Format (return only JSON):
{{
  "Information-Accuracy": [1-5],
  "Structural-Clarity": [1-5],
  "Faithfulness": [1-5]
}}
"""

In [17]:
# Load Original or Resume From Processed Copy
os.makedirs(processed_folder, exist_ok=True)

if os.path.exists(processed_csv):
    print("Existing processed file found. Resuming from previous progress...")
    df = pd.read_csv(processed_csv)
else:
    print("No processed copy found. Creating one...")
    df = pd.read_csv(original_csv)

    # Add scoring columns if missing
    score_cols = [
        "Information-Accuracy",
        "Structural-Clarity",
        "Faithfulness",
    ]
    for c in score_cols:
        if c not in df.columns:
            df[c] = ""

    df.to_csv(processed_csv, index=False, encoding="utf-8-sig")

print("Loaded rows:", len(df))

No processed copy found. Creating one...
Loaded rows: 1000


In [18]:
# Identify Rows That Need Processing
mask = df["Information-Accuracy"].isna() | (df["Information-Accuracy"] == "")
remaining_rows = df[mask]

start_index = len(df) - len(remaining_rows)
print(f"Starting evaluation from row {start_index}/{len(df)}\n")

batch_counter = 0

Starting evaluation from row 0/1000



#### MAIN LOOP

In [19]:
from tqdm import tqdm
import re
import json

# MAIN EVALUATION LOOP WITH TQDM
for idx in tqdm(remaining_rows.index, desc="Evaluating rows", unit="row"):

    row = df.loc[idx]

    prompt_filled = EVAL_PROMPT.format(
        conversation_history=row["conversation_history"],
        generated_summary=row["generated_summary"],
    )

    try:
        # SEND REQUEST TO CLAUDE
        response = anthropic.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=300,
            temperature=0,
            messages=[{"role": "user", "content": prompt_filled}],
        )

        # EXTRACT RAW TEXT
        raw_text = response.content[0].text.strip()
        if not raw_text:
            raise ValueError("Claude returned an empty response")

        # CLEAN THE JSON TEXT
        cleaned = raw_text.strip()

        cleaned = re.sub(r"^```[a-zA-Z0-9]*\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)
        cleaned = re.sub(r"<[^>]+>", "", cleaned).strip()
        cleaned = cleaned.replace("\ufeff", "").replace("\u200b", "").strip("`").strip()

        # TRY PARSING JSON
        try:
            result_json = json.loads(cleaned)
        except json.JSONDecodeError:
            raise ValueError(f"Invalid JSON from Claude:\n{raw_text}")

        # UPDATE DF ONLY IF SUCCESSFUL
        df.at[idx, "Information-Accuracy"] = result_json["Information-Accuracy"]
        df.at[idx, "Structural-Clarity"]   = result_json["Structural-Clarity"]
        df.at[idx, "Faithfulness"]         = result_json["Faithfulness"]

        # SAVE BATCH (SAFE)
        batch_counter += 1

        if batch_counter >= batch_size:
            df.to_csv(processed_csv, index=False, encoding="utf-8-sig")
            print(f"Batch saved (processed up to row {idx}).")
            batch_counter = 0

    except Exception as e:
        # ON ANY ERROR, STOP IMMEDIATELY
        # DO NOT SAVE ANYTHING
        print(f"\nERROR at row {idx}: {e}")
        print("Stopping execution WITHOUT saving this partial batch.")
        break

    time.sleep(batch_pause)

Evaluating rows:   5%|▍         | 49/1000 [02:49<1:24:10,  5.31s/row]

Batch saved (processed up to row 49).


Evaluating rows:  10%|▉         | 99/1000 [05:42<57:20,  3.82s/row]

Batch saved (processed up to row 99).


Evaluating rows:  15%|█▍        | 149/1000 [08:22<38:33,  2.72s/row]

...

Evaluating rows:  95%|█████████▍| 949/1000 [51:11<02:54,  3.43s/row]

Batch saved (processed up to row 949).


Evaluating rows: 100%|█████████▉| 999/1000 [54:05<00:03,  3.81s/row]

Batch saved (processed up to row 999).


Evaluating rows: 100%|██████████| 1000/1000 [54:08<00:00,  3.25s/row]


In [20]:
from datasets import Dataset

repo = "Lakshan2003/gemini-2.5-flash-customerservice-context-summarization-llm-judge-data"

# Convert to HF Dataset (remove index column if exists)
hf_dataset = Dataset.from_pandas(df.reset_index(drop=True))

# Push to the hub (creates parquet automatically)
hf_dataset.push_to_hub(repo, private=False)


readme_text = """
# customer-service-context-summarization-evaluation-data
Lakshan2003/gemini-2.5-flash-customerservice-context-summarization-llm-judge-data
Dataset updated with context summarization evaluation columns.
This README refresh triggers Hugging Face metadata re-index.
"""

with open("README.md", "w") as f:
    f.write(readme_text)

from huggingface_hub import HfApi

api = HfApi()
repo_id = "Lakshan2003/gemini-2.5-flash-customerservice-context-summarization-llm-judge-data"

api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Force metadata refresh – updated README only"
)

print("README uploaded. Metadata will refresh shortly.")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  79%|#######8  | 1.54MB / 1.96MB            

README.md:   0%|          | 0.00/784 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:9662: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


README uploaded. Metadata will refresh shortly.


In [21]:
from datasets import load_dataset
import pandas as pd

# Load dataset from Hugging Face
dataset = load_dataset(
    "Lakshan2003/gemini-2.5-flash-customerservice-context-summarization-llm-judge-data",
    split="train"
)

# Convert to pandas DataFrame
df = dataset.to_pandas()

# Correct column names (exact match)
task_columns = [
    "Information-Accuracy",
    "Structural-Clarity",
    "Faithfulness",
]

# Ensure numeric dtype
df[task_columns] = df[task_columns].apply(pd.to_numeric, errors="coerce")

# Compute task-wise mean
task_wise_mean = df[task_columns].mean()

# Convert to clean table
task_wise_mean_df = task_wise_mean.reset_index()
task_wise_mean_df.columns = ["Task", "Mean Score"]

README.md:   0%|          | 0.00/264 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [22]:
print(task_wise_mean_df)

                   Task  Mean Score
0  Information-Accuracy       4.793
1    Structural-Clarity       4.930
2          Faithfulness       4.770
